# EDA desde cero: CIS a Esclerosis Múltiple

Este notebook se enfoca solamente en la exploración de datos y en los hallazgos más importantes.

Por ahora no guarda resultados ni exporta tablas finales. La meta en esta etapa es entender el dataset.

## 1. Librerías

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style='whitegrid', palette='crest')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

## 2. Cargar el dataset

In [2]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = ROOT / 'data' / 'ms_dataset.csv'

raw_df = pd.read_csv(DATA_PATH)
print('Ruta:', DATA_PATH)
print('Filas, columnas:', raw_df.shape)
raw_df.head()

Ruta: D:\Repositorios\proyecto_final_admin_datos\data\ms_dataset.csv
Filas, columnas: (273, 20)


,Unnamed: 0,Gender,Age,Schooling,Breastfeeding,Varicella,Initial_Symptom,Mono_or_Polysymptomatic,Oligoclonal_Bands,LLSSEP,ULSSEP,VEP,BAEP,Periventricular_MRI,Cortical_MRI,Infratentorial_MRI,Spinal_Cord_MRI,Initial_EDSS,Final_EDSS,group
0,0,1,34,20.0,1,1,2.0,1,0,1,1,0,0,0,1,0,1,1.0,1.0,1
1,1,1,61,25.0,3,2,10.0,2,1,1,0,1,0,0,0,0,1,2.0,2.0,1
2,2,1,22,20.0,3,1,3.0,1,1,0,0,0,0,0,1,0,0,1.0,1.0,1
3,3,2,41,15.0,1,1,7.0,2,1,0,1,1,0,1,1,0,0,1.0,1.0,1
4,4,2,34,20.0,2,1,6.0,2,0,1,0,0,0,1,0,0,0,1.0,1.0,1


## 3. Diccionario de códigos

En este análisis mantenemos las columnas con su codificación original.

- `group`: `1 = CDMS`, `2 = non-CDMS`
- `Gender`: `1 = male`, `2 = female`
- `Breastfeeding`: `1 = yes`, `2 = no`, `3 = unknown`
- `Varicella`: `1 = positive`, `2 = negative`, `3 = unknown`
- `Mono_or_Polysymptomatic`: `1 = mono`, `2 = poly`, `3 = unknown`
- `Oligoclonal_Bands`: `0 = negative`, `1 = positive`, `2 = unknown`

## 4. Primera lectura del dataset

In [96]:
raw_df.columns.tolist()

['Unnamed: 0',
 'Gender',
 'Age',
 'Schooling',
 'Breastfeeding',
 'Varicella',
 'Initial_Symptom',
 'Mono_or_Polysymptomatic',
 'Oligoclonal_Bands',
 'LLSSEP',
 'ULSSEP',
 'VEP',
 'BAEP',
 'Periventricular_MRI',
 'Cortical_MRI',
 'Infratentorial_MRI',
 'Spinal_Cord_MRI',
 'Initial_EDSS',
 'Final_EDSS',
 'group']

In [97]:
pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'n_unique': raw_df.nunique(dropna=True)
})

,dtype,n_unique
Unnamed: 0,int64,273
Gender,int64,2
Age,int64,48
Schooling,float64,12
Breastfeeding,int64,3
Varicella,int64,3
Initial_Symptom,float64,15
Mono_or_Polysymptomatic,int64,3
Oligoclonal_Bands,int64,3
LLSSEP,int64,2


### Observaciones iniciales

- `Unnamed: 0` parece una columna índice.
- `group` es el target original.
- varias columnas son categóricas codificadas con números.
- algunas columnas usan una categoría `unknown`.

## 5. Calidad de datos

In [98]:
quality = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing': raw_df.isna().sum(),
    'missing_pct': (raw_df.isna().mean() * 100).round(2),
    'n_unique': raw_df.nunique(dropna=True)
}).sort_values(['missing_pct', 'n_unique'], ascending=[False, False])
quality

,dtype,missing,missing_pct,n_unique
Initial_EDSS,float64,148,54.21,3
Final_EDSS,float64,148,54.21,3
Initial_Symptom,float64,1,0.37,15
Schooling,float64,1,0.37,12
Unnamed: 0,int64,0,0.00,273
Age,int64,0,0.00,48
Breastfeeding,int64,0,0.00,3
Varicella,int64,0,0.00,3
Mono_or_Polysymptomatic,int64,0,0.00,3
Oligoclonal_Bands,int64,0,0.00,3


In [99]:
for col in raw_df.columns:
    print(f'\n[{col}]')
    print(sorted(raw_df[col].dropna().unique().tolist()))


[Unnamed: 0]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 

### Hallazgos de calidad

- `Initial_EDSS` y `Final_EDSS` tienen muchos faltantes.
- `Schooling` e `Initial_Symptom` tienen muy pocos faltantes.
- `Breastfeeding`, `Varicella`, `Mono_or_Polysymptomatic` y `Oligoclonal_Bands` contienen valores `unknown` codificados.

## 6. Revisar el target

In [100]:
raw_df['group'].value_counts().rename({1: 'CDMS', 2: 'non-CDMS'})

group
non-CDMS    148
CDMS        125
Name: count, dtype: int64

## 7. Columnas con `unknown`

In [101]:
unknown_map = {
    'Breastfeeding': 3,
    'Varicella': 3,
    'Mono_or_Polysymptomatic': 3,
    'Oligoclonal_Bands': 2,
}

for col, code in unknown_map.items():
    print(f'\n{col}')
    print(raw_df[col].value_counts(dropna=False).sort_index())
    print('unknown_count =', (raw_df[col] == code).sum())


Breastfeeding
Breastfeeding
1    131
2     57
3     85
Name: count, dtype: int64
unknown_count = 85

Varicella
Varicella
1    124
2    104
3     45
Name: count, dtype: int64
unknown_count = 45

Mono_or_Polysymptomatic
Mono_or_Polysymptomatic
1     81
2    186
3      6
Name: count, dtype: int64
unknown_count = 6

Oligoclonal_Bands
Oligoclonal_Bands
0    186
1     76
2     11
Name: count, dtype: int64
unknown_count = 11


### Hallazgo

Estas columnas requieren atención porque `unknown` no representa una categoría clínica real, sino un dato no disponible.

## 8. Leakage en `Initial_EDSS` y `Final_EDSS`

`Leakage` significa que una variable puede darle al modelo una pista artificial sobre el target.

In [102]:
pd.crosstab(raw_df['group'], raw_df['Initial_EDSS'].notna(), normalize='index')

Initial_EDSS,False,True
group,,
1,0.0,1.0
2,1.0,0.0


In [103]:
pd.crosstab(raw_df['group'], raw_df['Final_EDSS'].notna(), normalize='index')

Final_EDSS,False,True
group,,
1,0.0,1.0
2,1.0,0.0


### Hallazgo

`Initial_EDSS` y `Final_EDSS` no solo tienen faltantes; además, el patrón de faltantes depende demasiado del target. Por eso son candidatas a excluirse del modelado.

## 9. Relación con el target

In [104]:
pd.crosstab(raw_df['Oligoclonal_Bands'], raw_df['group'], normalize='columns').round(3)

group,1,2
Oligoclonal_Bands,,
0,0.528,0.811
1,0.472,0.115
2,0.000,0.074


In [105]:
pd.crosstab(raw_df['Breastfeeding'], raw_df['group'], normalize='columns').round(3)

group,1,2
Breastfeeding,,
1,0.520,0.446
2,0.264,0.162
3,0.216,0.392


In [106]:
corr_df = raw_df.drop(columns=['Unnamed: 0']).corr(numeric_only=True)
corr_df['group'].sort_values(ascending=False)

group                      1.000000
Gender                     0.240633
Varicella                  0.169099
Breastfeeding              0.142580
Mono_or_Polysymptomatic   -0.034895
BAEP                      -0.052089
Age                       -0.064426
Spinal_Cord_MRI           -0.120647
Schooling                 -0.184484
Oligoclonal_Bands         -0.186351
ULSSEP                    -0.194231
VEP                       -0.215663
LLSSEP                    -0.221406
Cortical_MRI              -0.222184
Initial_Symptom           -0.389021
Infratentorial_MRI        -0.425954
Periventricular_MRI       -0.541345
Initial_EDSS                    NaN
Final_EDSS                      NaN
Name: group, dtype: float64

In [107]:
mi_df = raw_df.drop(columns=['Unnamed: 0']).copy()
mi_df = mi_df.drop(columns=['Initial_EDSS', 'Final_EDSS'])
mi_df['Schooling'] = mi_df['Schooling'].fillna(mi_df['Schooling'].median())
mi_df['Initial_Symptom'] = mi_df['Initial_Symptom'].fillna(-1)

X = mi_df.drop(columns=['group'])
y = mi_df['group']

mi_scores = mutual_info_classif(X, y, discrete_features=True, random_state=42)
pd.DataFrame({'feature': X.columns, 'mutual_information': mi_scores}).sort_values('mutual_information', ascending=False).head(15)

,feature,mutual_information
12,Periventricular_MRI,0.155151
5,Initial_Symptom,0.120988
1,Age,0.107520
7,Oligoclonal_Bands,0.098499
14,Infratentorial_MRI,0.094187
2,Schooling,0.042815
4,Varicella,0.042234
0,Gender,0.029115
13,Cortical_MRI,0.024823
8,LLSSEP,0.024642


### Hallazgos principales

- las variables MRI aparecen como señales importantes
- `Age` también parece aportar información
- `Oligoclonal_Bands` merece atención por su relación con el target
- `Initial_EDSS` y `Final_EDSS` no deben interpretarse como buenas variables predictoras por el problema de leakage

## 10. Feature engineering propuesto

Se identificaron variables nuevas útiles para una etapa posterior:

- `mri_lesion_count`
- `has_any_mri_lesion`
- `evoked_positive_count`
- `has_any_evoked_positive`
- `age_group`

Estas variables resumen mejor la información clínica y son fáciles de justificar.